In [1]:
!pip -q install requests pandas numpy astropy astroquery lightkurve tqdm matplotlib

In [2]:
import json
import math
import re
import zipfile
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import lightkurve as lk

warnings.filterwarnings("ignore")

ROOT = Path("/content/exointel_real_data")
DATA_DIR = ROOT / "data"
LC_DIR = DATA_DIR / "lightcurves"

DATA_DIR.mkdir(parents=True, exist_ok=True)
LC_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COUNT = 200

# For speed, do not try all 200 light curves first.
# Start with 25. Later change to 50, 100, then 200.
MAX_REAL_LIGHTCURVES_TO_ATTEMPT = 25

MAX_PRODUCTS_PER_SEARCH = 2
MAX_POINTS_PER_LIGHTCURVE = 1600
PHASE_WINDOW = 0.16

print("Working folder:", ROOT)
print("Data folder:", DATA_DIR)
print("Lightcurve folder:", LC_DIR)
print("Target count requested:", TARGET_COUNT)
print("Real light curves to attempt first:", MAX_REAL_LIGHTCURVES_TO_ATTEMPT)

Working folder: /content/exointel_real_data
Data folder: /content/exointel_real_data/data
Lightcurve folder: /content/exointel_real_data/data/lightcurves
Target count requested: 200
Real light curves to attempt first: 25


/usr/local/lib/python3.12/dist-packages/lightkurve/prf/__init__.py:7: UserWarning: Warning: the tpfmodel submodule is not available without oktopus installed, which requires a current version of autograd. See #1452 for details.
  warnings.warn(


In [3]:
TAP_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

ADQL_QUERY = f"""
SELECT TOP {TARGET_COUNT}
  pl_name,
  hostname,
  sy_snum,
  sy_pnum,
  ra,
  dec,
  pl_orbper,
  pl_orbsmax,
  pl_ratror,
  pl_rade,
  pl_bmasse,
  pl_orbincl,
  pl_orbeccen,
  pl_trandep,
  pl_trandur,
  pl_tranmid,
  st_teff,
  st_rad,
  st_mass,
  st_logg,
  st_met,
  disc_year,
  discoverymethod
FROM pscomppars
WHERE tran_flag = 1
  AND pl_name IS NOT NULL
  AND hostname IS NOT NULL
  AND pl_orbper IS NOT NULL
  AND pl_orbsmax IS NOT NULL
  AND pl_ratror IS NOT NULL
  AND pl_orbincl IS NOT NULL
  AND pl_orbeccen IS NOT NULL
  AND st_rad IS NOT NULL
  AND st_teff IS NOT NULL
  AND pl_trandep IS NOT NULL
ORDER BY pl_trandep DESC
"""

payload = {
    "request": "doQuery",
    "lang": "ADQL",
    "format": "json",
    "query": ADQL_QUERY,
}

response = requests.post(TAP_URL, data=payload, timeout=60)
response.raise_for_status()

rows = response.json()
df = pd.DataFrame(rows)

print("Rows downloaded:", len(df))
df.head()

Rows downloaded: 200


,pl_name,hostname,sy_snum,sy_pnum,ra,dec,pl_orbper,pl_orbsmax,pl_ratror,pl_rade,...,pl_trandep,pl_trandur,pl_tranmid,st_teff,st_rad,st_mass,st_logg,st_met,disc_year,discoverymethod
0,WTS-2 b,WTS-2,1,1,293.732783,36.815494,1.018707,0.01855,0.18630,15.277867,...,9.155864,1.494787,2.454318e+06,5000.0,0.752,0.820,4.600,0.20,2014,Transit
1,HATS-6 b,HATS-6,1,1,88.146808,-19.031627,3.325273,0.03623,0.17978,11.186582,...,3.230000,2.040960,2.456644e+06,3724.0,0.570,0.574,4.683,0.20,2015,Transit
2,Kepler-45 b,Kepler-45,1,1,292.872929,41.064173,2.455239,0.03000,0.17900,10.760000,...,3.204000,1.727220,2.455004e+06,3820.0,0.550,0.590,4.700,0.28,2011,Transit
3,CoRoT-18 b,CoRoT-18,1,1,98.172417,-0.031605,1.900069,0.02950,0.13410,14.680000,...,2.571701,2.387000,2.455322e+06,5440.0,1.000,0.950,4.400,-0.10,2011,Transit
4,WASP-43 b,WASP-43,1,1,154.908187,-9.806443,0.813475,0.01420,0.15940,10.424000,...,2.550000,1.159200,2.455529e+06,4400.0,0.600,0.580,4.650,-0.05,2011,Transit


In [4]:
def slugify(value):
    text = str(value or "unknown-target").strip().lower()
    text = text.replace("+", " plus ")
    text = re.sub(r"[’'\"]", "", text)
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = re.sub(r"^-+|-+$", "", text)
    return text or "unknown-target"


def clean_number(x):
    if pd.isna(x):
        return None
    try:
        y = float(x)
        return y if math.isfinite(y) else None
    except Exception:
        return None


def clean_int(x):
    y = clean_number(x)
    return int(y) if y is not None else None


def clean_text(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None


targets = []

for _, row in df.iterrows():
    planet = clean_text(row.get("pl_name"))
    host = clean_text(row.get("hostname"))
    depth_percent = clean_number(row.get("pl_trandep"))
    depth_ppm = depth_percent * 10000.0 if depth_percent is not None else None

    item = {
        "pl_name": planet,
        "hostname": host,
        "sy_snum": clean_int(row.get("sy_snum")),
        "sy_pnum": clean_int(row.get("sy_pnum")),
        "ra": clean_number(row.get("ra")),
        "dec": clean_number(row.get("dec")),
        "pl_orbper": clean_number(row.get("pl_orbper")),
        "pl_orbsmax": clean_number(row.get("pl_orbsmax")),
        "pl_ratror": clean_number(row.get("pl_ratror")),
        "pl_rade": clean_number(row.get("pl_rade")),
        "pl_bmasse": clean_number(row.get("pl_bmasse")),
        "pl_orbincl": clean_number(row.get("pl_orbincl")),
        "pl_orbeccen": clean_number(row.get("pl_orbeccen")),
        "pl_trandep": depth_ppm,
        "pl_trandep_percent": depth_percent,
        "pl_trandur": clean_number(row.get("pl_trandur")),
        "pl_tranmid": clean_number(row.get("pl_tranmid")),
        "st_teff": clean_number(row.get("st_teff")),
        "st_rad": clean_number(row.get("st_rad")),
        "st_mass": clean_number(row.get("st_mass")),
        "st_logg": clean_number(row.get("st_logg")),
        "st_met": clean_number(row.get("st_met")),
        "disc_year": clean_int(row.get("disc_year")),
        "discoverymethod": clean_text(row.get("discoverymethod")) or "Transit",
        "lightcurve_file": f"{slugify(planet)}.json",
        "lightcurve_available": False,
    }

    if item["pl_name"] and item["hostname"]:
        targets.append(item)

cache = {
    "schema": "exointel-prime-gold-target-cache-v3",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "source": "NASA Exoplanet Archive TAP pscomppars",
    "tap_url": TAP_URL,
    "adql": ADQL_QUERY.strip(),
    "target_count": len(targets),
    "lightcurve_directory": "data/lightcurves",
    "columns": list(targets[0].keys()) if targets else [],
    "targets": targets,
}

EXOPLANET_JSON = DATA_DIR / "exoplanets.json"
EXOPLANET_JSON.write_text(json.dumps(cache, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Saved:", EXOPLANET_JSON)
print("Target count:", len(targets))
print("First 5 targets:")
for t in targets[:5]:
    print(t["pl_name"], "| depth:", round(t["pl_trandep"], 1), "ppm", "| file:", t["lightcurve_file"])

Saved: /content/exointel_real_data/data/exoplanets.json
Target count: 200
First 5 targets:
WTS-2 b | depth: 91558.6 ppm | file: wts-2-b.json
HATS-6 b | depth: 32300.0 ppm | file: hats-6-b.json
Kepler-45 b | depth: 32040.0 ppm | file: kepler-45-b.json
CoRoT-18 b | depth: 25717.0 ppm | file: corot-18-b.json
WASP-43 b | depth: 25500.0 ppm | file: wasp-43-b.json


In [6]:
def robust_median_normalize(flux):
    flux = np.asarray(flux, dtype=float)
    finite = np.isfinite(flux)

    if finite.sum() < 10:
        return flux * np.nan

    med = np.nanmedian(flux[finite])

    if not np.isfinite(med) or med <= 0:
        return flux * np.nan

    return flux / med


def sigma_clip_phase_flux(phase, flux, sigma=6.0):
    phase = np.asarray(phase, dtype=float)
    flux = np.asarray(flux, dtype=float)

    mask = np.isfinite(phase) & np.isfinite(flux) & (flux > 0.2) & (flux < 1.8)
    phase = phase[mask]
    flux = flux[mask]

    if len(flux) < 30:
        return phase, flux

    med = np.nanmedian(flux)
    mad = np.nanmedian(np.abs(flux - med))

    if not np.isfinite(mad) or mad <= 0:
        return phase, flux

    robust_sigma = 1.4826 * mad
    keep = np.abs(flux - med) < max(0.02, sigma * robust_sigma)
    return phase[keep], flux[keep]


def wrap_phase(x):
    return ((x + 0.5) % 1.0) - 0.5


def choose_epoch_and_fold(time, flux, period, t0_archive=None):
    time = np.asarray(time, dtype=float)
    flux = np.asarray(flux, dtype=float)

    finite = np.isfinite(time) & np.isfinite(flux)
    time = time[finite]
    flux = flux[finite]

    if len(time) < 30:
        return np.array([]), np.array([])

    candidates = []

    if t0_archive is not None and np.isfinite(t0_archive):
        candidates += [
            float(t0_archive),
            float(t0_archive) - 2457000.0,
            float(t0_archive) - 2454833.0,
            float(t0_archive) - 2400000.5,
        ]

    candidates.append(float(np.nanmedian(time)))

    best_t0 = candidates[0]
    best_score = np.inf

    for t0 in candidates:
        phase = wrap_phase((time - t0) / period)
        idx = np.argsort(flux)[:max(8, len(flux) // 100)]
        centre = np.nanmedian(phase[idx])
        score = abs(centre)

        if np.isfinite(score) and score < best_score:
            best_score = score
            best_t0 = t0

    phase = wrap_phase((time - best_t0) / period)

    idx = np.argsort(flux)[:max(8, len(flux) // 100)]
    centre = np.nanmedian(phase[idx])

    if np.isfinite(centre):
        phase = wrap_phase(phase - centre)

    return phase, flux


def median_bin(phase, flux, max_points=1600):
    phase = np.asarray(phase, dtype=float)
    flux = np.asarray(flux, dtype=float)

    mask = np.isfinite(phase) & np.isfinite(flux)
    phase = phase[mask]
    flux = flux[mask]

    if len(phase) <= max_points:
        order = np.argsort(phase)
        return phase[order], flux[order]

    edges = np.linspace(np.nanmin(phase), np.nanmax(phase), max_points + 1)
    b_phase = []
    b_flux = []

    for i in range(max_points):
        m = (phase >= edges[i]) & (phase < edges[i + 1])
        if m.sum() == 0:
            continue
        b_phase.append(np.nanmedian(phase[m]))
        b_flux.append(np.nanmedian(flux[m]))

    return np.array(b_phase), np.array(b_flux)


def save_lightcurve_json(target, phase, flux, source_note):
    planet = target["pl_name"]
    filename = target["lightcurve_file"]
    path = LC_DIR / filename

    points = [
        {
            "phase": round(float(p), 8),
            "flux": round(float(f), 8),
            "error": None,
        }
        for p, f in zip(phase, flux)
        if np.isfinite(p) and np.isfinite(f)
    ]

    payload = {
        "schema": "exointel-prime-real-lightcurve-v1",
        "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source": source_note,
        "planet": planet,
        "hostname": target["hostname"],
        "period_days": target["pl_orbper"],
        "transit_midpoint": target.get("pl_tranmid"),
        "points_count": len(points),
        "phase": [p["phase"] for p in points],
        "flux": [p["flux"] for p in points],
        "error": [None for _ in points],
        "points": points,
    }

    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path, len(points)

In [7]:
def safe_search_lightcurves(name):
    name = str(name).strip()
    results = []

    search_configs = [
        {"mission": "Kepler", "author": "Kepler"},
        {"mission": "Kepler"},
        {"mission": "K2", "author": "K2SFF"},
        {"mission": "K2", "author": "EVEREST"},
        {"mission": "K2"},
        {"mission": "TESS", "author": "SPOC"},
        {"mission": "TESS", "author": "QLP"},
        {"mission": "TESS", "author": "TESS-SPOC"},
    ]

    for cfg in search_configs:
        try:
            sr = lk.search_lightcurve(name, **cfg)
            if len(sr) > 0:
                results.append((cfg, sr))
                print(f"  Found {len(sr)} products for '{name}' with {cfg}")
        except Exception as exc:
            print(f"  Search skipped for '{name}' with {cfg}: {exc}")

    return results


def get_flux_from_lc(lc):
    try:
        if hasattr(lc, "pdcsap_flux") and lc.pdcsap_flux is not None:
            flux = np.asarray(lc.pdcsap_flux.value, dtype=float)
            source = "PDCSAP_FLUX"
        else:
            flux = np.asarray(lc.flux.value, dtype=float)
            source = "FLUX"
    except Exception:
        flux = np.asarray(lc.flux.value, dtype=float)
        source = "FLUX"

    return flux, source


def fetch_lightcurve_for_target(target, phase_window=0.16, max_points=1600, max_products_per_search=2):
    planet = target["pl_name"]
    host = target["hostname"]
    period = target.get("pl_orbper")
    t0 = target.get("pl_tranmid")

    if period is None or not np.isfinite(period) or period <= 0:
        return False, "invalid period"

    search_names = []

    if isinstance(planet, str) and planet.strip():
        search_names.append(planet)

    if isinstance(host, str) and host.strip() and host not in search_names:
        search_names.append(host)

    for name in search_names:
        print(f"\nSearching MAST for: {name}")
        search_results = safe_search_lightcurves(name)

        if not search_results:
            continue

        for cfg, sr in search_results:
            n_try = min(len(sr), max_products_per_search)

            for i in range(n_try):
                try:
                    print(f"  Downloading product {i+1}/{n_try} from {cfg}")
                    lc = sr[i].download()

                    if lc is None:
                        continue

                    lc = lc.remove_nans()

                    try:
                        if hasattr(lc, "quality") and lc.quality is not None:
                            q = np.asarray(lc.quality, dtype=int)
                            lc = lc[q == 0]
                    except Exception:
                        pass

                    time = np.asarray(lc.time.value, dtype=float)
                    flux, flux_source = get_flux_from_lc(lc)

                    if len(time) < 80 or len(flux) < 80:
                        continue

                    flux = robust_median_normalize(flux)
                    phase, flux = choose_epoch_and_fold(time, flux, float(period), t0)
                    phase, flux = sigma_clip_phase_flux(phase, flux)

                    crop = (phase >= -phase_window) & (phase <= phase_window)
                    phase_c = phase[crop]
                    flux_c = flux[crop]

                    if len(phase_c) < 60:
                        crop = (phase >= -0.30) & (phase <= 0.30)
                        phase_c = phase[crop]
                        flux_c = flux[crop]

                    if len(phase_c) < 60:
                        print("  Product rejected: not enough points around transit")
                        continue

                    phase_b, flux_b = median_bin(phase_c, flux_c, max_points=max_points)

                    if len(phase_b) < 60:
                        print("  Product rejected after binning")
                        continue

                    note = f"MAST real light curve via Lightkurve; search='{name}', config={cfg}, flux='{flux_source}'"
                    path, n = save_lightcurve_json(target, phase_b, flux_b, note)

                    return True, f"saved {n} points to {path.name}"

                except Exception as exc:
                    print(f"  Product failed: {exc}")
                    continue

    return False, "no usable real MAST light curve found"

In [8]:
priority_planets = [
    "HD 189733 b",
    "WASP-43 b",
    "HAT-P-7 b",
    "Kepler-10 b",
    "Kepler-45 b",
    "TrES-3 b",
    "GJ 1214 b",
    "WASP-12 b",
    "55 Cnc e",
]


def find_target_by_name(name, targets):
    name_low = name.lower()
    for t in targets:
        if str(t.get("pl_name", "")).lower() == name_low:
            return t
    return None


priority_targets = []

for name in priority_planets:
    t = find_target_by_name(name, targets)
    if t is not None:
        priority_targets.append(t)

seen = set(t["pl_name"] for t in priority_targets)

for t in targets[:50]:
    if t["pl_name"] not in seen:
        priority_targets.append(t)
        seen.add(t["pl_name"])

print("Priority test list:")
for t in priority_targets[:20]:
    print(" ", t["pl_name"], "|", t["hostname"])

successes = []

for target in tqdm(priority_targets[:20], desc="Testing likely real light-curve targets"):
    print("\n" + "=" * 90)
    print("Trying:", target["pl_name"], "| Host:", target["hostname"])

    ok, msg = fetch_lightcurve_for_target(
        target,
        phase_window=PHASE_WINDOW,
        max_points=MAX_POINTS_PER_LIGHTCURVE,
        max_products_per_search=MAX_PRODUCTS_PER_SEARCH,
    )

    print("Result:", ok, msg)

    if ok:
        target["lightcurve_available"] = True
        successes.append(target["pl_name"])
        break
    else:
        target["lightcurve_available"] = False

print("\nSuccessful test targets:", successes)
print("Generated files:", [p.name for p in sorted(LC_DIR.glob('*.json'))])

Priority test list:
  HD 189733 b | HD 189733
  WASP-43 b | WASP-43
  Kepler-45 b | Kepler-45
  WTS-2 b | WTS-2
  HATS-6 b | HATS-6
  CoRoT-18 b | CoRoT-18
  Kepler-428 b | Kepler-428
  Kepler-17 b | Kepler-17
  HAT-P-18 b | HAT-P-18
  HAT-P-37 b | HAT-P-37
  TOI-2406 b | TOI-2406
  WASP-67 b | WASP-67
  Kepler-75 b | Kepler-75
  WTS-1 b | WTS-1
  KOI-217 b | KOI-217
  WASP-23 b | WASP-23
  WASP-59 b | WASP-59
  OGLE-TR-111 b | OGLE-TR-111
  Kepler-30 c | Kepler-30
  WASP-31 b | WASP-31


Testing likely real light-curve targets:   0%|          | 0/20 [00:00<?, ?it/s]


Trying: HD 189733 b | Host: HD 189733

Searching MAST for: HD 189733 b
  Found 6 products for 'HD 189733 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 3 products for 'HD 189733 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 3 products for 'HD 189733 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 1580 points to hd-189733-b.json

Successful test targets: ['HD 189733 b']
Generated files: ['hd-189733-b.json']


In [9]:
success_count = 0
failed = []

for idx, target in enumerate(tqdm(targets[:MAX_REAL_LIGHTCURVES_TO_ATTEMPT], desc="Building real light curves"), start=1):
    planet = target["pl_name"]
    existing_path = LC_DIR / target["lightcurve_file"]

    print("\n" + "=" * 90)
    print(f"[{idx}/{MAX_REAL_LIGHTCURVES_TO_ATTEMPT}] Target:", planet, "| Host:", target["hostname"])

    if existing_path.exists():
        print("Already exists:", existing_path.name)
        target["lightcurve_available"] = True
        success_count += 1
        continue

    ok, msg = fetch_lightcurve_for_target(
        target,
        phase_window=PHASE_WINDOW,
        max_points=MAX_POINTS_PER_LIGHTCURVE,
        max_products_per_search=MAX_PRODUCTS_PER_SEARCH,
    )

    print("Result:", ok, msg)

    if ok:
        target["lightcurve_available"] = True
        success_count += 1
    else:
        target["lightcurve_available"] = False
        failed.append((planet, msg))

    cache["generated_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    cache["targets"] = targets
    EXOPLANET_JSON.write_text(json.dumps(cache, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("\nFinished.")
print("Successful light curves:", success_count)
print("Failed:", len(failed))
failed[:10]

Building real light curves:   0%|          | 0/25 [00:00<?, ?it/s]


[1/25] Target: WTS-2 b | Host: WTS-2

Searching MAST for: WTS-2 b
  Found 8 products for 'WTS-2 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 8 products for 'WTS-2 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 1600 points to wts-2-b.json

[2/25] Target: HATS-6 b | Host: HATS-6

Searching MAST for: HATS-6 b
  Found 8 products for 'HATS-6 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 5 products for 'HATS-6 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 3 products for 'HATS-6 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 1600 points to hats-6-b.json

[3/25] Target: Kepler-45 b | Host: Kepler-45

Searching MAST for: Kepler-45 b
  Found 49 products for 'Kepler-45 b' with {'mission': 'Kepler', 'author': 'Kepler'}
  Found 50 products for 'Kepler-45 b' with {'mission': 'Kepler'}
  Found 8 products for 'Kepler-45 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 8 products for 'Kepler-45 b' with {'mission': 'TESS', 'auth

Result: True saved 1515 points to hat-p-27-b.json

Finished.
Successful light curves: 25
Failed: 0


[]

In [15]:
# CELL 8B — Continue from existing downloaded light curves, do not repeat old files

ADDITIONAL_ATTEMPTS = 150   # change to 175 if you want to attempt all remaining targets

existing_files = {p.name for p in LC_DIR.glob("*.json")}

pending_targets = []

for target in targets:
    file_name = target["lightcurve_file"]
    file_path = LC_DIR / file_name

    if file_path.exists():
        target["lightcurve_available"] = True
    else:
        pending_targets.append(target)

print("Existing real light-curve files:", len(existing_files))
print("Pending targets still without local light curves:", len(pending_targets))
print("This run will attempt:", min(ADDITIONAL_ATTEMPTS, len(pending_targets)))

success_count = 0
failed_more = []

for idx, target in enumerate(
    tqdm(pending_targets[:ADDITIONAL_ATTEMPTS], desc="Downloading remaining real light curves"),
    start=1
):
    planet = target["pl_name"]
    host = target["hostname"]
    file_name = target["lightcurve_file"]
    file_path = LC_DIR / file_name

    print("\n" + "=" * 90)
    print(f"[{idx}/{min(ADDITIONAL_ATTEMPTS, len(pending_targets))}] Target:", planet, "| Host:", host)

    if file_path.exists():
        print("Already exists, skipping:", file_name)
        target["lightcurve_available"] = True
        continue

    ok, msg = fetch_lightcurve_for_target(
        target,
        phase_window=PHASE_WINDOW,
        max_points=MAX_POINTS_PER_LIGHTCURVE,
        max_products_per_search=MAX_PRODUCTS_PER_SEARCH,
    )

    print("Result:", ok, msg)

    if ok:
        target["lightcurve_available"] = True
        success_count += 1
    else:
        target["lightcurve_available"] = False
        failed_more.append((planet, msg))

    # Save progress after every target so you do not lose work if Colab disconnects
    cache["generated_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
    cache["target_count"] = len(targets)
    cache["targets"] = targets
    EXOPLANET_JSON.write_text(
        json.dumps(cache, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8"
    )

print("\nContinuation batch finished.")
print("New successful light curves in this run:", success_count)
print("Total light-curve files now:", len(list(LC_DIR.glob('*.json'))))
print("Failed in this batch:", len(failed_more))
failed_more[:20]

Existing real light-curve files: 25
Pending targets still without local light curves: 175
This run will attempt: 150



[1/150] Target: K2-237 b | Host: K2-237

Searching MAST for: K2-237 b
  Found 2 products for 'K2-237 b' with {'mission': 'K2', 'author': 'K2SFF'}
  Found 2 products for 'K2-237 b' with {'mission': 'K2', 'author': 'EVEREST'}
  Found 6 products for 'K2-237 b' with {'mission': 'K2'}
  Found 4 products for 'K2-237 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 3 products for 'K2-237 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 2 products for 'K2-237 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 319 points to k2-237-b.json

[2/150] Target: HAT-P-53 b | Host: HAT-P-53

Searching MAST for: HAT-P-53 b
  Found 1 products for 'HAT-P-53 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 2 products for 'HAT-P-53 b' with {'mission': 'TESS', 'author': 'QLP'}
Result: True saved 1600 points to hat-p-53-b.json

[3/150] Target: HAT-P-45 b | Host: HAT-P-45

Searching MAST for: HAT-P-45 b
  Found 1 products for 'HAT-P-45 b' with {'mission': 'TESS', 'author': 'SPO

Result: True saved 1600 points to hat-p-41-b.json

[10/150] Target: HAT-P-22 b | Host: HAT-P-22

Searching MAST for: HAT-P-22 b
  Found 4 products for 'HAT-P-22 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 3 products for 'HAT-P-22 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 3 products for 'HAT-P-22 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 1600 points to hat-p-22-b.json

[11/150] Target: WASP-21 b | Host: WASP-21

Searching MAST for: WASP-21 b
  Found 4 products for 'WASP-21 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 2 products for 'WASP-21 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 2 products for 'WASP-21 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 1600 points to wasp-21-b.json

[12/150] Target: Kepler-422 b | Host: Kepler-422

Searching MAST for: Kepler-422 b
  Found 54 products for 'Kepler-422 b' with {'mission': 'Kepler', 'author': 'Kepler'}
  Found 55 products for 'Kepler-422 b' with {'mis

Result: True saved 1314 points to wasp-24-b.json

[16/150] Target: WASP-14 b | Host: WASP-14

Searching MAST for: WASP-14 b
  Found 1 products for 'WASP-14 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 1 products for 'WASP-14 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 1 products for 'WASP-14 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}


Result: True saved 1419 points to wasp-14-b.json

[17/150] Target: HAT-P-39 b | Host: HAT-P-39

Searching MAST for: HAT-P-39 b
  Found 5 products for 'HAT-P-39 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 5 products for 'HAT-P-39 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 5 products for 'HAT-P-39 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 1600 points to hat-p-39-b.json

[18/150] Target: Kepler-77 b | Host: Kepler-77

Searching MAST for: Kepler-77 b
  Found 30 products for 'Kepler-77 b' with {'mission': 'Kepler', 'author': 'Kepler'}
  Found 31 products for 'Kepler-77 b' with {'mission': 'Kepler'}
  Found 7 products for 'Kepler-77 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 9 products for 'Kepler-77 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 7 products for 'Kepler-77 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 1600 points to kepler-77-b.json

[19/150] Target: WASP-70 A b | Host: WASP-70 A

Searchi

Result: True saved 1557 points to corot-8-b.json

[28/150] Target: Kepler-817 b | Host: Kepler-817

Searching MAST for: Kepler-817 b
  Found 11 products for 'Kepler-817 b' with {'mission': 'Kepler', 'author': 'Kepler'}
  Found 12 products for 'Kepler-817 b' with {'mission': 'Kepler'}
  Found 5 products for 'Kepler-817 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 5 products for 'Kepler-817 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 806 points to kepler-817-b.json

[29/150] Target: HAT-P-2 b | Host: HAT-P-2

Searching MAST for: HAT-P-2 b
  Found 8 products for 'HAT-P-2 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 6 products for 'HAT-P-2 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 6 products for 'HAT-P-2 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 1600 points to hat-p-2-b.json

[30/150] Target: KOI-94 d | Host: KOI-94

Searching MAST for: KOI-94 d
  Found 41 products for 'KOI-94 d' with {'mission': 'Kepler', '

Result: True saved 1559 points to corot-3-b.json

[37/150] Target: Kepler-297 c | Host: Kepler-297

Searching MAST for: Kepler-297 c
  Found 36 products for 'Kepler-297 c' with {'mission': 'Kepler', 'author': 'Kepler'}
  Found 37 products for 'Kepler-297 c' with {'mission': 'Kepler'}
  Found 7 products for 'Kepler-297 c' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 7 products for 'Kepler-297 c' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 642 points to kepler-297-c.json

[38/150] Target: Kepler-63 b | Host: Kepler-63

Searching MAST for: Kepler-63 b
  Found 63 products for 'Kepler-63 b' with {'mission': 'Kepler', 'author': 'Kepler'}
  Found 64 products for 'Kepler-63 b' with {'mission': 'Kepler'}
  Found 16 products for 'Kepler-63 b' with {'mission': 'TESS', 'author': 'SPOC'}
  Found 12 products for 'Kepler-63 b' with {'mission': 'TESS', 'author': 'QLP'}
  Found 12 products for 'Kepler-63 b' with {'mission': 'TESS', 'author': 'TESS-SPOC'}
Result: True saved 

[('WASP-38 b', 'no usable real MAST light curve found')]

In [16]:
cache["generated_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
cache["target_count"] = len(targets)
cache["targets"] = targets

EXOPLANET_JSON.write_text(json.dumps(cache, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Updated:", EXOPLANET_JSON)
print("Target count:", len(targets))
print("Lightcurve JSON files:", len(list(LC_DIR.glob("*.json"))))

available = [t["pl_name"] for t in targets if t.get("lightcurve_available")]
print("Targets with real light curves:", available[:20])

Updated: /content/exointel_real_data/data/exoplanets.json
Target count: 200
Lightcurve JSON files: 174
Targets with real light curves: ['WTS-2 b', 'HATS-6 b', 'Kepler-45 b', 'CoRoT-18 b', 'WASP-43 b', 'HD 189733 b', 'Kepler-428 b', 'Kepler-17 b', 'HAT-P-18 b', 'HAT-P-37 b', 'TOI-2406 b', 'WASP-67 b', 'Kepler-75 b', 'WTS-1 b', 'KOI-217 b', 'WASP-23 b', 'WASP-59 b', 'OGLE-TR-111 b', 'Kepler-30 c', 'WASP-31 b']


In [17]:
cache["generated_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
cache["target_count"] = len(targets)
cache["targets"] = targets

EXOPLANET_JSON.write_text(json.dumps(cache, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Updated:", EXOPLANET_JSON)
print("Target count:", len(targets))
print("Lightcurve JSON files:", len(list(LC_DIR.glob("*.json"))))

available = [t["pl_name"] for t in targets if t.get("lightcurve_available")]
print("Targets with real light curves:", available[:20])

Updated: /content/exointel_real_data/data/exoplanets.json
Target count: 200
Lightcurve JSON files: 174
Targets with real light curves: ['WTS-2 b', 'HATS-6 b', 'Kepler-45 b', 'CoRoT-18 b', 'WASP-43 b', 'HD 189733 b', 'Kepler-428 b', 'Kepler-17 b', 'HAT-P-18 b', 'HAT-P-37 b', 'TOI-2406 b', 'WASP-67 b', 'Kepler-75 b', 'WTS-1 b', 'KOI-217 b', 'WASP-23 b', 'WASP-59 b', 'OGLE-TR-111 b', 'Kepler-30 c', 'WASP-31 b']


In [18]:
cache["generated_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
cache["target_count"] = len(targets)
cache["targets"] = targets

EXOPLANET_JSON.write_text(json.dumps(cache, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print("Updated:", EXOPLANET_JSON)
print("Target count:", len(targets))
print("Lightcurve JSON files:", len(list(LC_DIR.glob("*.json"))))

available = [t["pl_name"] for t in targets if t.get("lightcurve_available")]
print("Targets with real light curves:", available[:20])

Updated: /content/exointel_real_data/data/exoplanets.json
Target count: 200
Lightcurve JSON files: 174
Targets with real light curves: ['WTS-2 b', 'HATS-6 b', 'Kepler-45 b', 'CoRoT-18 b', 'WASP-43 b', 'HD 189733 b', 'Kepler-428 b', 'Kepler-17 b', 'HAT-P-18 b', 'HAT-P-37 b', 'TOI-2406 b', 'WASP-67 b', 'Kepler-75 b', 'WTS-1 b', 'KOI-217 b', 'WASP-23 b', 'WASP-59 b', 'OGLE-TR-111 b', 'Kepler-30 c', 'WASP-31 b']


In [19]:
ZIP_PATH = Path("/content/exointel_real_data_package.zip")

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for path in DATA_DIR.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(ROOT))

print("Created ZIP:", ZIP_PATH)
print("Files inside:")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for name in z.namelist()[:40]:
        print(" ", name)

Created ZIP: /content/exointel_real_data_package.zip
Files inside:
  data/exoplanets.json
  data/lightcurves/kepler-817-b.json
  data/lightcurves/hat-p-27-b.json
  data/lightcurves/hat-p-42-b.json
  data/lightcurves/kepler-145-c.json
  data/lightcurves/toi-2406-b.json
  data/lightcurves/hip-41378-b.json
  data/lightcurves/hat-p-45-b.json
  data/lightcurves/kepler-425-b.json
  data/lightcurves/kepler-27-c.json
  data/lightcurves/kepler-199-c.json
  data/lightcurves/kepler-415-b.json
  data/lightcurves/koi-217-b.json
  data/lightcurves/kepler-17-b.json
  data/lightcurves/corot-18-b.json
  data/lightcurves/koi-94-d.json
  data/lightcurves/kepler-58-d.json
  data/lightcurves/hat-p-22-b.json
  data/lightcurves/kepler-300-c.json
  data/lightcurves/kepler-31-c.json
  data/lightcurves/kepler-328-b.json
  data/lightcurves/kepler-426-b.json
  data/lightcurves/kepler-33-e.json
  data/lightcurves/kepler-595-b.json
  data/lightcurves/hd-18599-b.json
  data/lightcurves/kepler-644-b.json
  data/light

In [20]:
from google.colab import files

files.download(str(ZIP_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>